# Appliance Classifier — multi-label ONNX replacement for `extract_appliances()`

Trains a local EfficientNet-B0 multi-label classifier on the 8 appliance categories  
tracked by JKKTrackr. Exports to ONNX — identical inference path as `classifier.onnx`.

**Images**: `training/labeled/appliances/` — 157 images  
**Labels**: 8 binary classes (multiple can be true per image)  
**Step 1**: Label images interactively (skip if labels already saved)  
**Step 2**: Train EfficientNet-B0  
**Step 3**: Evaluate per-class  
**Step 4**: Export to ONNX

In [ ]:
import os, sys, json, io, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import onnxruntime as ort

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False
    print('pip install ipywidgets  ← needed for the labeling UI')

# Device
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:', DEVICE)

# The 8 appliance classes (must match listings.appliances field values)
APPLIANCE_CLASSES = [
    'エアコン',
    '床暖房',
    '追い焚き',
    '浴室乾燥',
    'オートロック',
    '温水洗浄便座',
    'モニター付き',
    'エレベーター',
]
N_CLASSES = len(APPLIANCE_CLASSES)
print('Classes:', APPLIANCE_CLASSES)

In [ ]:
# ── Load appliance images ──────────────────────────────────────────────────
LABELED_DIR  = Path('training/labeled/appliances')
LABELS_CSV   = Path('training/appliance_labels.csv')  # multi-label CSV (this notebook writes it)

image_paths = sorted(LABELED_DIR.glob('*.jpg')) + sorted(LABELED_DIR.glob('*.png'))
print(f'Found {len(image_paths)} appliance images')

# Sample grid
N_SAMPLE = min(20, len(image_paths))
cols = 5
rows = (N_SAMPLE + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3))
for ax, p in zip(axes.flat, image_paths[:N_SAMPLE]):
    ax.imshow(Image.open(p).convert('RGB'))
    ax.set_title(p.name[:22], fontsize=6)
    ax.axis('off')
for ax in axes.flat[N_SAMPLE:]:
    ax.axis('off')
plt.suptitle('Appliance images to label', fontsize=11)
plt.tight_layout()
plt.show()

## Step 1 — Label images

Run the cell below to open the interactive labeling UI.  
**Check all appliances visible in the photo**, then click **Save & Next →**.  
Labels auto-save to `training/appliance_labels.csv` after each image.

If `LABELS_CSV` already exists and is complete, skip to Step 2.

In [ ]:
# ── Load existing labels if any ───────────────────────────────────────────
labels_dict = {}  # path_str → list[int] (binary vector length N_CLASSES)

if LABELS_CSV.exists():
    df_existing = pd.read_csv(LABELS_CSV)
    for _, row in df_existing.iterrows():
        labels_dict[row['path']] = [int(row[c]) for c in APPLIANCE_CLASSES]
    print(f'Loaded {len(labels_dict)} existing labels from {LABELS_CSV}')
else:
    print('No existing labels — start labeling below')

In [ ]:
# ── Interactive labeling UI ────────────────────────────────────────────────
# Shows one image at a time with checkboxes for each appliance class.
# Saves after every click so you can close and resume anytime.

def save_labels():
    rows = []
    for pstr, vec in labels_dict.items():
        row = {'path': pstr}
        row.update(dict(zip(APPLIANCE_CLASSES, vec)))
        rows.append(row)
    pd.DataFrame(rows).to_csv(LABELS_CSV, index=False)


def make_labeling_ui():
    if not WIDGETS_OK:
        print('ipywidgets not available')
        return

    # Start from first unlabeled image
    unlabeled = [p for p in image_paths if str(p) not in labels_dict]
    queue = unlabeled if unlabeled else list(image_paths)
    current = [0]

    if not queue:
        print('All images already labeled!')
        return

    progress = widgets.HTML()
    img_widget = widgets.Image(format='jpeg', width=380)
    checks = {c: widgets.Checkbox(value=False, description=c, style={'description_width': 'initial'})
              for c in APPLIANCE_CLASSES}
    btn_next   = widgets.Button(description='Save & Next →', button_style='primary')
    btn_skip   = widgets.Button(description='Skip (no appliance visible)', button_style='warning')
    btn_prev   = widgets.Button(description='← Back')
    status_out = widgets.Output()

    def show(idx):
        p = queue[idx]
        img = Image.open(p).convert('RGB')
        img.thumbnail((380, 380))
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=90)
        img_widget.value = buf.getvalue()
        progress.value = f'<b>{idx+1}/{len(queue)}</b> — {p.name} &nbsp; ({len(labels_dict)} saved)'
        existing = labels_dict.get(str(p), None)
        for i, (c, cb) in enumerate(checks.items()):
            cb.value = bool(existing[i]) if existing else False

    def save_current():
        p = queue[current[0]]
        labels_dict[str(p)] = [int(cb.value) for cb in checks.values()]
        save_labels()

    def on_next(b):
        save_current()
        with status_out:
            clear_output()
        if current[0] + 1 < len(queue):
            current[0] += 1
            show(current[0])
        else:
            progress.value = f'<b>Done! {len(labels_dict)} images labeled.</b>'

    def on_skip(b):
        for cb in checks.values():
            cb.value = False
        save_current()
        on_next(b)

    def on_prev(b):
        if current[0] > 0:
            current[0] -= 1
            show(current[0])

    btn_next.on_click(on_next)
    btn_skip.on_click(on_skip)
    btn_prev.on_click(on_prev)
    show(0)

    half = N_CLASSES // 2
    cb_left  = widgets.VBox(list(checks.values())[:half])
    cb_right = widgets.VBox(list(checks.values())[half:])
    cb_panel = widgets.HBox([cb_left, cb_right])
    btns     = widgets.HBox([btn_prev, btn_next, btn_skip])
    ui = widgets.VBox([progress, img_widget, cb_panel, btns, status_out])
    display(ui)

make_labeling_ui()

In [ ]:
# ── Label distribution ─────────────────────────────────────────────────────
df = pd.read_csv(LABELS_CSV)
print(f'Labeled images: {len(df)}')
print(f'Unlabeled:      {len(image_paths) - len(df)}')
print()
counts = df[APPLIANCE_CLASSES].sum().sort_values(ascending=False)
print('Positive examples per class:')
print(counts.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
counts.plot(kind='bar', ax=ax, color='#1976d2', edgecolor='white')
ax.set_ylabel('# images with this appliance')
ax.set_title('Appliance class distribution in labeled set')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Step 2 — Train

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
class ApplianceDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        img = self.transform(img)
        label = torch.tensor([float(row[c]) for c in APPLIANCE_CLASSES])
        return img, label


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
val_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

df = pd.read_csv(LABELS_CSV)
n_val = max(1, int(len(df) * 0.2))
df_val   = df.sample(n=n_val, random_state=42)
df_train = df.drop(df_val.index)

ds_train = ApplianceDataset(df_train, train_tf)
ds_val   = ApplianceDataset(df_val, val_tf)
dl_train = DataLoader(ds_train, batch_size=16, shuffle=True,  num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=16, shuffle=False, num_workers=0)
print(f'Train: {len(ds_train)}, Val: {len(ds_val)}')

In [ ]:
# ── Model — EfficientNet-B0 with multi-label head ─────────────────────────
backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
in_features = backbone.classifier[1].in_features
backbone.classifier[1] = nn.Linear(in_features, N_CLASSES)
model = backbone.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────
# Phase 1: freeze backbone, only train the new head (5 epochs)
# Phase 2: unfreeze all, fine-tune with lower LR (15 epochs)

criterion = nn.BCEWithLogitsLoss()

def train_epoch(loader, opt):
    model.train()
    total_loss, n = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(imgs)
        n += len(imgs)
    return total_loss / n

@torch.no_grad()
def val_loss(loader):
    model.eval()
    total, n = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        total += criterion(model(imgs), labels).item() * len(imgs)
        n += len(imgs)
    return total / n


# Phase 1 — head only
for p in model.features.parameters():
    p.requires_grad = False
opt1 = torch.optim.Adam(model.classifier.parameters(), lr=3e-4)

print('Phase 1 — training head only')
hist = {'train': [], 'val': []}
for ep in range(5):
    tl = train_epoch(dl_train, opt1)
    vl = val_loss(dl_val)
    hist['train'].append(tl)
    hist['val'].append(vl)
    print(f'  Epoch {ep+1:2d}/5  train={tl:.4f}  val={vl:.4f}')

# Phase 2 — full fine-tune
for p in model.parameters():
    p.requires_grad = True
opt2 = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15)

print('\nPhase 2 — full fine-tune')
best_val, best_state = float('inf'), None
for ep in range(15):
    tl = train_epoch(dl_train, opt2)
    vl = val_loss(dl_val)
    sched.step()
    hist['train'].append(tl)
    hist['val'].append(vl)
    flag = ''
    if vl < best_val:
        best_val = vl
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = ' ← best'
    print(f'  Epoch {ep+1:2d}/15  train={tl:.4f}  val={vl:.4f}{flag}')

model.load_state_dict(best_state)
print(f'\nRestored best checkpoint (val={best_val:.4f})')

# Loss curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist['train'], label='train')
ax.plot(hist['val'],   label='val')
ax.axvline(5, color='grey', linestyle='--', linewidth=0.8, label='phase 2 start')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Training loss')
ax.legend()
plt.tight_layout()
plt.show()

## Step 3 — Evaluate

In [ ]:
# ── Per-class precision / recall / F1 on validation set ───────────────────
THRESHOLD = 0.40  # sigmoid output threshold for positive prediction

@torch.no_grad()
def predict_all(loader):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE)).cpu()
        all_logits.append(logits)
        all_labels.append(labels)
    return torch.cat(all_logits).sigmoid().numpy(), torch.cat(all_labels).numpy()

probs, truth = predict_all(dl_val)
preds = (probs >= THRESHOLD).astype(int)

# Per-class metrics
rows = []
for i, cls in enumerate(APPLIANCE_CLASSES):
    tp = ((preds[:, i] == 1) & (truth[:, i] == 1)).sum()
    fp = ((preds[:, i] == 1) & (truth[:, i] == 0)).sum()
    fn = ((preds[:, i] == 0) & (truth[:, i] == 1)).sum()
    prec  = tp / (tp + fp) if (tp + fp) else 0.0
    rec   = tp / (tp + fn) if (tp + fn) else 0.0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    n_pos = int(truth[:, i].sum())
    rows.append({'class': cls, 'precision': prec, 'recall': rec, 'F1': f1, 'n_positive': n_pos})

metrics_df = pd.DataFrame(rows).set_index('class')
print(f'Threshold = {THRESHOLD}')
print(metrics_df.round(3).to_string())

# Visualise
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(N_CLASSES)
w = 0.28
ax.bar(x - w, metrics_df['precision'], w, label='Precision', color='#1976d2')
ax.bar(x,     metrics_df['recall'],    w, label='Recall',    color='#43a047')
ax.bar(x + w, metrics_df['F1'],        w, label='F1',        color='#fb8c00')
ax.set_xticks(x)
ax.set_xticklabels(APPLIANCE_CLASSES, rotation=25, ha='right')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.7)
ax.set_title('Per-class precision / recall / F1')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold sweep — pick the best threshold per-class or globally ────────
thresholds = np.linspace(0.1, 0.9, 17)
f1_by_thresh = []

for t in thresholds:
    p = (probs >= t).astype(int)
    f1s = []
    for i in range(N_CLASSES):
        tp = ((p[:, i] == 1) & (truth[:, i] == 1)).sum()
        fp = ((p[:, i] == 1) & (truth[:, i] == 0)).sum()
        fn = ((p[:, i] == 0) & (truth[:, i] == 1)).sum()
        pr = tp / (tp + fp) if (tp + fp) else 0
        re = tp / (tp + fn) if (tp + fn) else 0
        f1s.append(2 * pr * re / (pr + re) if (pr + re) else 0)
    f1_by_thresh.append(np.mean(f1s))

best_t = thresholds[np.argmax(f1_by_thresh)]
print(f'Best global threshold: {best_t:.2f}  (mean F1 = {max(f1_by_thresh):.3f})')

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(thresholds, f1_by_thresh, marker='o')
ax.axvline(best_t, color='red', linestyle='--', linewidth=0.8, label=f'best={best_t:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Mean F1')
ax.set_title('Mean F1 vs. decision threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Example predictions on validation images ───────────────────────────────
N_SHOW = 12
val_paths = df_val['path'].tolist()

cols = 4
rows = (N_SHOW + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))

for ax_idx, (ax, img_path) in enumerate(zip(axes.flat, val_paths[:N_SHOW])):
    img_pil = Image.open(img_path).convert('RGB')
    img_t = val_tf(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob_vec = model(img_t).cpu().sigmoid().numpy()[0]

    row = df_val[df_val['path'] == img_path].iloc[0]
    gt = [APPLIANCE_CLASSES[i] for i in range(N_CLASSES) if row[APPLIANCE_CLASSES[i]]]
    pred = [(APPLIANCE_CLASSES[i], prob_vec[i]) for i in range(N_CLASSES) if prob_vec[i] >= best_t]

    ax.imshow(img_pil)
    ax.axis('off')
    gt_str   = ', '.join(gt) or 'none'
    pred_str = ', '.join(f'{n}({p:.0%})' for n, p in pred) or 'none'
    ax.set_title(f'GT:   {gt_str}\nPred: {pred_str}', fontsize=6.5, loc='left',
                 color='#333')

for ax in axes.flat[N_SHOW:]:
    ax.axis('off')

plt.suptitle('Validation predictions (GT vs. Model)', fontsize=11)
plt.tight_layout()
plt.show()

## Step 4 — Export to ONNX

In [ ]:
# ── ONNX export ────────────────────────────────────────────────────────────
OUT_PT   = Path('training/appliance_classifier.pt')
OUT_ONNX = Path('training/appliance_classifier.onnx')
META_JSON = Path('training/appliance_classifier_meta.json')

model.eval().cpu()
dummy = torch.zeros(1, 3, 224, 224)

torch.export.export(model, (dummy,))
torch.onnx.export(
    model, dummy, str(OUT_ONNX),
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
torch.save(model.state_dict(), str(OUT_PT))

# Save metadata (class list + threshold) alongside the ONNX model
meta = {'classes': APPLIANCE_CLASSES, 'threshold': float(best_t), 'input_size': 224}
META_JSON.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

print(f'ONNX: {OUT_ONNX}  ({OUT_ONNX.stat().st_size // 1024} KB)')
print(f'PT:   {OUT_PT}')
print(f'Meta: {META_JSON}')

In [ ]:
# ── Smoke-test ONNX inference ──────────────────────────────────────────────
sess = ort.InferenceSession(str(OUT_ONNX), providers=['CPUExecutionProvider'])

test_img_path = val_paths[0]
img_t = val_tf(Image.open(test_img_path).convert('RGB')).unsqueeze(0).numpy()
logits = sess.run(['logits'], {'image': img_t})[0][0]
probs_onnx = 1 / (1 + np.exp(-logits))  # sigmoid

detected = [(APPLIANCE_CLASSES[i], float(probs_onnx[i]))
            for i in range(N_CLASSES) if probs_onnx[i] >= best_t]

print('ONNX smoke test →', test_img_path)
print('Detected appliances:', detected or 'none')
print('\nAll class probabilities:')
for cls, p in zip(APPLIANCE_CLASSES, probs_onnx):
    bar = '█' * int(p * 20)
    print(f'  {cls:12s} {p:.3f}  {bar}')

In [ ]:
# ── Drop-in replacement for extract_appliances() ──────────────────────────
# Copy this into image_pipeline.py, place appliance_classifier.onnx and
# appliance_classifier_meta.json next to classifier.onnx.

REPLACEMENT_CODE = '''
# ── load once at module level ──────────────────────────────────────────────
_APPLIANCE_SESSION = None
_APPLIANCE_META    = None

def _load_appliance_classifier():
    global _APPLIANCE_SESSION, _APPLIANCE_META
    if _APPLIANCE_SESSION is not None:
        return
    import json
    model_path = os.path.join(os.path.dirname(__file__), "appliance_classifier.onnx")
    meta_path  = os.path.join(os.path.dirname(__file__), "appliance_classifier_meta.json")
    if not os.path.exists(model_path):
        log.warning("appliance_classifier.onnx not found — appliance extraction disabled")
        return
    import onnxruntime as ort
    _APPLIANCE_SESSION = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    if os.path.exists(meta_path):
        _APPLIANCE_META = json.loads(Path(meta_path).read_text())
    log.info("Appliance classifier loaded")


def extract_appliances(image_path: str) -> list[str]:
    """
    Identify appliances in an image using local ONNX multi-label classifier.
    Returns list of appliance names (Japanese), empty list on failure.
    """
    import struct
    import numpy as np
    from PIL import Image

    _load_appliance_classifier()
    if _APPLIANCE_SESSION is None:
        return []

    classes   = _APPLIANCE_META["classes"] if _APPLIANCE_META else APPLIANCE_CLASSES
    threshold = _APPLIANCE_META["threshold"] if _APPLIANCE_META else 0.4
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    try:
        img = Image.open(image_path).convert("RGB")
        img = img.resize((224, 224), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        arr = (arr - MEAN) / STD
        tensor = arr.transpose(2, 0, 1)[None].astype(np.float32)  # [1,3,224,224]
        logits = _APPLIANCE_SESSION.run(["logits"], {"image": tensor})[0][0]
        probs  = 1 / (1 + np.exp(-logits))  # sigmoid
        return [classes[i] for i in range(len(classes)) if probs[i] >= threshold]
    except Exception as e:
        log.warning(f"Appliance extraction failed for {image_path}: {e}")
        return []
'''

print(REPLACEMENT_CODE)